In [ ]:
!pip install gymnasium
!pip install datasets
!pip install transformers
!pip install tensorflow
!pip install scikit-learn

In [1]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
from keras import layers

import numpy as np
import tensorflow as tf
import gymnasium as gym
import scipy.signal

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [2]:
# PPO hyperparameters
steps_per_epoch = 4000
epochs = 30
gamma = 0.99
clip_ratio = 0.2
policy_learning_rate = 3e-4
value_function_learning_rate = 1e-3
train_policy_iterations = 80
train_value_iterations = 80
lam = 0.97
target_kl = 0.01
hidden_sizes = (64, 64)
render = False

In [3]:
# PPO Implementation referrenced from https://keras.io/examples/rl/ppo_cartpole/

def discounted_cumulative_sums(x, discount):
    # Discounted cumulative sums of vectors for computing rewards-to-go and advantage estimates
    return scipy.signal.lfilter([1], [1, float(-discount)], x[::-1], axis=0)[::-1]


class Buffer:
    # Buffer for storing trajectories
    def __init__(self, observation_dimensions, size, gamma=0.99, lam=0.95):
        # Buffer initialization
        self.observation_buffer = np.zeros(
            (size, observation_dimensions), dtype=np.float32
        )
        self.action_buffer = np.zeros(size, dtype=np.int32)
        self.advantage_buffer = np.zeros(size, dtype=np.float32)
        self.reward_buffer = np.zeros(size, dtype=np.float32)
        self.return_buffer = np.zeros(size, dtype=np.float32)
        self.value_buffer = np.zeros(size, dtype=np.float32)
        self.logprobability_buffer = np.zeros(size, dtype=np.float32)
        self.gamma, self.lam = gamma, lam
        self.pointer, self.trajectory_start_index = 0, 0

    def store(self, observation, action, reward, value, logprobability):
        # Append one step of agent-environment interaction
        self.observation_buffer[self.pointer] = observation
        self.action_buffer[self.pointer] = action
        self.reward_buffer[self.pointer] = reward
        self.value_buffer[self.pointer] = value
        self.logprobability_buffer[self.pointer] = logprobability
        self.pointer += 1

    def finish_trajectory(self, last_value=0):
        # Finish the trajectory by computing advantage estimates and rewards-to-go
        path_slice = slice(self.trajectory_start_index, self.pointer)
        rewards = np.append(self.reward_buffer[path_slice], last_value)
        values = np.append(self.value_buffer[path_slice], last_value)

        deltas = rewards[:-1] + self.gamma * values[1:] - values[:-1]

        self.advantage_buffer[path_slice] = discounted_cumulative_sums(
            deltas, self.gamma * self.lam
        )
        self.return_buffer[path_slice] = discounted_cumulative_sums(
            rewards, self.gamma
        )[:-1]

        self.trajectory_start_index = self.pointer

    def get(self):
        # Get all data of the buffer and normalize the advantages
        self.pointer, self.trajectory_start_index = 0, 0
        advantage_mean, advantage_std = (
            np.mean(self.advantage_buffer),
            np.std(self.advantage_buffer),
        )
        self.advantage_buffer = (self.advantage_buffer - advantage_mean) / advantage_std
        return (
            self.observation_buffer,
            self.action_buffer,
            self.advantage_buffer,
            self.return_buffer,
            self.logprobability_buffer,
        )


def mlp(x, sizes, activation=keras.activations.tanh, output_activation=None):
    # Build a feedforward neural network
    for size in sizes[:-1]:
        x = layers.Dense(units=size, activation=activation)(x)
    return layers.Dense(units=sizes[-1], activation=output_activation)(x)


def logprobabilities(logits, a, n_actions):
    logprobabilities_all = keras.ops.log_softmax(logits)
    logprobability = keras.ops.sum(
        keras.ops.one_hot(a, tf.cast(n_actions, dtype=tf.int32)) * logprobabilities_all, axis=1
    )
    return logprobability


seed_generator = keras.random.SeedGenerator(1337)


# Sample action from actor
@tf.function
def sample_action(observation, actor):
    logits = actor(observation)
    action = keras.ops.squeeze(
        keras.random.categorical(logits, 1, seed=seed_generator), axis=1
    )
    return logits, action


# Train the policy by maxizing the PPO-Clip objective
@tf.function
def train_policy(observation_buffer, action_buffer, logprobability_buffer,
                 advantage_buffer, actor_model, optimizer, n_actions, clip_val):

    with tf.GradientTape() as tape:
        # Use the passed actor_model and n_actions
        new_logits = actor_model(observation_buffer)
        new_logprobs = logprobabilities(new_logits, action_buffer, n_actions)

        ratio = keras.ops.exp(new_logprobs - logprobability_buffer)

        min_advantage = keras.ops.where(
            advantage_buffer > 0,
            (1 + clip_val) * advantage_buffer,
            (1 - clip_val) * advantage_buffer,
        )

        policy_loss = -keras.ops.mean(
            keras.ops.minimum(ratio * advantage_buffer, min_advantage)
        )

    policy_grads = tape.gradient(policy_loss, actor_model.trainable_variables)
    optimizer.apply_gradients(zip(policy_grads, actor_model.trainable_variables))

    # Calculate KL for early stopping
    kl = keras.ops.mean(logprobability_buffer - new_logprobs)
    return keras.ops.sum(kl)


# Train the value function by regression on mean-squared error
@tf.function
def train_value_function(observation_buffer, return_buffer, critic, value_optimizer):
    with tf.GradientTape() as tape:  # Record operations for automatic differentiation.
        value_loss = keras.ops.mean((return_buffer - critic(observation_buffer)) ** 2)
    value_grads = tape.gradient(value_loss, critic.trainable_variables)
    value_optimizer.apply_gradients(zip(value_grads, critic.trainable_variables))

In [4]:
def run_ppo_experiment(epochs=30, hidden_sizes=(64, 64), clip_ratio=0.2):

  # Initialize the environment and get the dimensionality of the
  # observation space and the number of possible actions
  env = gym.make("CartPole-v1")
  observation_dimensions = env.observation_space.shape[0]
  num_actions = env.action_space.n

  # Initialize the buffer
  buffer = Buffer(observation_dimensions, steps_per_epoch)

  # Initialize the actor and the critic as keras models
  observation_input = keras.Input(shape=(observation_dimensions,), dtype="float32")
  logits = mlp(observation_input, list(hidden_sizes) + [num_actions])
  actor = keras.Model(inputs=observation_input, outputs=logits)
  value = keras.ops.squeeze(mlp(observation_input, list(hidden_sizes) + [1]), axis=1)
  critic = keras.Model(inputs=observation_input, outputs=value)

  # Initialize the policy and the value function optimizers
  policy_optimizer = keras.optimizers.Adam(learning_rate=policy_learning_rate)
  value_optimizer = keras.optimizers.Adam(learning_rate=value_function_learning_rate)

  # Initialize the observation, episode return and episode length
  observation, _ = env.reset()
  episode_return, episode_length = 0, 0


  for epoch in range(epochs):
      # Initialize the sum of the returns, lengths and number of episodes for each epoch
      sum_return = 0
      sum_length = 0
      num_episodes = 0

      # Iterate over the steps of each epoch
      for t in range(steps_per_epoch):
          if render:
              env.render()

          # Get the logits, action, and take one step in the environment
          observation = observation.reshape(1, -1)
          logits, action = sample_action(observation, actor)
          observation_new, reward, done, _, _ = env.step(action[0].numpy())
          episode_return += reward
          episode_length += 1

          # Get the value and log-probability of the action
          value_t = critic(observation)
          logprobability_t = logprobabilities(logits, action, 2)

          # Store obs, act, rew, v_t, logp_pi_t
          buffer.store(observation, action, reward, value_t, logprobability_t)

          # Update the observation
          observation = observation_new

          # Finish trajectory if reached to a terminal state
          terminal = done
          if terminal or (t == steps_per_epoch - 1):
              last_value = 0 if done else critic(observation.reshape(1, -1))
              buffer.finish_trajectory(last_value)
              sum_return += episode_return
              sum_length += episode_length
              num_episodes += 1
              observation, _ = env.reset()
              episode_return, episode_length = 0, 0

      # Get values from the buffer
      (
          observation_buffer,
          action_buffer,
          advantage_buffer,
          return_buffer,
          logprobability_buffer,
      ) = buffer.get()

      # Update the policy and implement early stopping using KL divergence
      for _ in range(train_policy_iterations):
          kl = train_policy(
            observation_buffer,
            action_buffer,
            logprobability_buffer,
            advantage_buffer,
            actor,             # Local actor
            policy_optimizer,  # Local optimizer
            num_actions,       # Local num_actions
            clip_ratio         # Local clip_ratio
          )
          print(f"DEBUG: Sub-iteration KL is {kl}")
      if kl > 1.5 * target_kl:
          break

      # Update the value function
      for _ in range(train_value_iterations):
          train_value_function(observation_buffer, return_buffer, critic, value_optimizer)

      # Print mean return and length for each epoch
      print(
          f" Epoch: {epoch + 1}. Mean Return: {sum_return / num_episodes}. Mean Length: {sum_length / num_episodes}"
      )

In [20]:
# Experiment 0: Running PPO with default parameters (epoch = 30, hidden_sizes = (64, 64), clip_ratio = 0.2)
run_ppo_experiment(epochs, hidden_sizes, clip_ratio)

 Epoch: 1. Mean Return: 20.94240837696335. Mean Length: 20.94240837696335
 Epoch: 2. Mean Return: 26.666666666666668. Mean Length: 26.666666666666668
 Epoch: 3. Mean Return: 35.08771929824562. Mean Length: 35.08771929824562
 Epoch: 4. Mean Return: 51.94805194805195. Mean Length: 51.94805194805195
 Epoch: 5. Mean Return: 70.17543859649123. Mean Length: 70.17543859649123
 Epoch: 6. Mean Return: 102.56410256410257. Mean Length: 102.56410256410257
 Epoch: 7. Mean Return: 137.93103448275863. Mean Length: 137.93103448275863
 Epoch: 8. Mean Return: 222.22222222222223. Mean Length: 222.22222222222223
 Epoch: 9. Mean Return: 250.0. Mean Length: 250.0
 Epoch: 10. Mean Return: 307.6923076923077. Mean Length: 307.6923076923077
 Epoch: 11. Mean Return: 444.44444444444446. Mean Length: 444.44444444444446
 Epoch: 12. Mean Return: 800.0. Mean Length: 800.0
 Epoch: 13. Mean Return: 2000.0. Mean Length: 2000.0
 Epoch: 14. Mean Return: 4000.0. Mean Length: 4000.0
 Epoch: 15. Mean Return: 4000.0. Mean Len

This intial experiment is meant to show how the PPO performs under the default parameters.


*   Epoch 1-8 shows a slow steady growth as the agent randomly explores, highlighting how advantage estimation is at work, with the critic pushing the agent towards better choices
*   Epoch 9-14 shows a significant increase in mean return and mean length, showing how gradient ascent adjusts the weights significantly towards the better outcome, once the initial exploration stage is complete
*   Epoch 14-30 show a steady results, with mean return consistently at 4000 and a generally high mean length. This shows how KL Divergence is keeping the agent within a "safe zone", ensuring that the agent does not adjust its weigths in too drastic of a way, helping to prevent policy collapse




In [5]:
# Experiment 1: Running PPO with decresed epoch parameters (epoch = 5, hidden_sizes = (64, 64), clip_ratio = 0.2)
run_ppo_experiment(5, hidden_sizes, clip_ratio)

 Epoch: 1. Mean Return: 21.50537634408602. Mean Length: 21.50537634408602
 Epoch: 2. Mean Return: 27.210884353741495. Mean Length: 27.210884353741495
 Epoch: 3. Mean Return: 35.39823008849557. Mean Length: 35.39823008849557
 Epoch: 4. Mean Return: 51.94805194805195. Mean Length: 51.94805194805195
 Epoch: 5. Mean Return: 72.72727272727273. Mean Length: 72.72727272727273


Experiment 1 shows that reducing the epochs to 5 provides a snapshot of the learning velocity. While 5 epochs are not enough to produce a "winner", they are enough to prove that the agent is improving meaninfully, with a gradual increase in mean return and mean length over the 5 epochs.

In [5]:
# Experiment 2: Running PPO with decresed epoch and increased hidden layers parameters (epoch = 5, hidden_sizes = (128, 128), clip_ratio = 0.2)
run_ppo_experiment(5, (128, 128), clip_ratio)

 Epoch: 1. Mean Return: 22.3463687150838. Mean Length: 22.3463687150838
 Epoch: 2. Mean Return: 26.143790849673202. Mean Length: 26.143790849673202
 Epoch: 3. Mean Return: 36.69724770642202. Mean Length: 36.69724770642202


Experiment 2 showed that increasing the hidden layers made the agent less stable. By doubling the hidden layers, this created more sensitivity in the agent, where small weight changes can result in large policy changes. This will quickly result in a large difference KL Divergence, causing early termination, which is why the experiment ran for only 3 epochs before being stopped early.

This is not enough to conclude that there is a improvement in convergence speed, since it is likely by epoch 3 the agent has not reached its stable state yet and its still actively exploring and learning.

In [5]:
# Experiment 3: Running PPO with decresed epoch and increased clip ratio parameters (epoch = 5, hidden_sizes = (64, 64), clip_ratio = 0.4)
run_ppo_experiment(5, hidden_sizes, 0.4)

DEBUG: Sub-iteration KL is 4.410743770222325e-09
DEBUG: Sub-iteration KL is -0.00038096398930065334
DEBUG: Sub-iteration KL is -0.0005372904124669731
DEBUG: Sub-iteration KL is -0.0004740669683087617
DEBUG: Sub-iteration KL is -0.00019839030574075878
DEBUG: Sub-iteration KL is 0.0002812157035805285
DEBUG: Sub-iteration KL is 0.000955309602431953
DEBUG: Sub-iteration KL is 0.0018143546767532825
DEBUG: Sub-iteration KL is 0.002828150289133191
DEBUG: Sub-iteration KL is 0.003979386296123266
DEBUG: Sub-iteration KL is 0.005276193842291832
DEBUG: Sub-iteration KL is 0.006713429000228643
DEBUG: Sub-iteration KL is 0.008273039013147354
DEBUG: Sub-iteration KL is 0.00993586890399456
DEBUG: Sub-iteration KL is 0.01166294515132904
DEBUG: Sub-iteration KL is 0.013425539247691631
DEBUG: Sub-iteration KL is 0.015188943594694138
DEBUG: Sub-iteration KL is 0.016929076984524727
DEBUG: Sub-iteration KL is 0.018607810139656067
DEBUG: Sub-iteration KL is 0.02020074427127838
DEBUG: Sub-iteration KL is 0.0

Experiment 3 failed to complete a single epoch despite successful sub-iterations in the policy update. The added logs show a high KL Divergence (up to 0.027), which is much higher than the typical target KL (0.002).

This shows that a clip ratio of 0.4 is too aggressive for the CartPole environment. While the agent attempts to learn quickly, the resulting policy shifts are too large for the Value Function to track, likely leading to gradient divergence or a mathematical crash.